<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/%D0%9A%D0%BE%D0%B4_%D0%B4%D0%BB%D1%8F_Google_Colab_(%D0%9D%D0%B0%D0%B4%D1%96%D0%B9%D0%BD%D0%B5_%D1%87%D0%B8%D1%82%D0%B0%D0%BD%D0%BD%D1%8F).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import os
import warnings
import io
import subprocess
import sys

# Функція для автоматичного встановлення відсутніх бібліотек
def install_dependencies():
    packages = ['xlsxwriter', 'python-calamine', 'openpyxl']
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            print(f"Встановлення необхідної бібліотеки: {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Встановлюємо залежності перед запуском
install_dependencies()

from google.colab import files

# Ігноруємо всі попередження щодо форматів
warnings.filterwarnings('ignore')

def ultra_robust_read(file_path):
    """
    Найпотужніший метод читання, який ігнорує XML помилки Microsoft.
    """
    # Метод 1: Пряме читання через двигун calamine (Rust-базований)
    try:
        return pd.read_excel(file_path, header=None, engine='calamine')
    except Exception:
        pass

    # Метод 2: Читання значень без аналізу метаданих через openpyxl (read_only режим)
    try:
        import openpyxl
        wb = openpyxl.load_workbook(file_path, data_only=True, read_only=True)
        sheet = wb.active
        data = [[cell.value for cell in row] for row in sheet.iter_rows()]
        return pd.DataFrame(data)
    except Exception:
        pass

    # Метод 3: Останній шанс - стандартний pandas
    try:
        return pd.read_excel(file_path, header=None)
    except Exception as e:
        raise Exception(f"Жоден метод не зміг прочитати цей файл Microsoft. Помилка: {e}")

def process_grading_xlsx():
    """
    Основний цикл: завантаження -> обхід XML-помилок -> сортування -> результат.
    """
    print("--- ЗАВАНТАЖЕННЯ ФАЙЛУ ---")
    uploaded = files.upload()

    if not uploaded:
        print("Файл не обрано.")
        return

    file_name = list(uploaded.keys())[0]

    try:
        print("Витягую дані (ігноруючи помилки XML)...")
        raw_df = ultra_robust_read(file_name)

        # Пошук заголовка "Повне ім’я"
        header_idx = None
        for i, row in raw_df.iterrows():
            row_vals = [str(v).strip().lower() for v in row if v is not None]
            if any("повне ім’я" in s or "повне ім'я" in s for s in row_vals):
                header_idx = i
                break

        if header_idx is None:
            print("ПОМИЛКА: Не знайдено заголовок 'Повне ім’я'.")
            return

        # Формуємо таблицю
        headers = raw_df.iloc[header_idx].astype(str).str.strip().tolist()
        df = raw_df.iloc[header_idx + 1:].copy()
        df.columns = headers

        # Видаляємо пусті стовпці
        df = df.loc[:, ~df.columns.str.contains('None|nan|^$')]

    except Exception as e:
        print(f"Критична помилка при обробці: {e}")
        return

    # Пошук ключових стовпців (динамічно)
    name_col = None
    score_col = None

    for col in df.columns:
        col_name = str(col).lower()
        if 'повне ім' in col_name:
            name_col = col
        if 'бал' in col_name or 'оцін' in col_name:
            score_col = col

    if not name_col:
        print(f"Стовпець 'Повне ім'я' не знайдено серед: {list(df.columns)}")
        return

    # Сортування
    df = df.sort_values(by=name_col).reset_index(drop=True)

    output_file = 'Оброблена_відомість_final.xlsx'

    # Створення результату з формулами SUMIF
    writer = pd.ExcelWriter(output_file, engine='xlsxwriter')
    df.to_excel(writer, index=False, sheet_name='Оцінки')

    workbook  = writer.book
    worksheet = writer.sheets['Оцінки']

    cols = list(df.columns)
    name_letter = chr(65 + cols.index(name_col))

    # Визначаємо букву для балів
    if score_col in cols:
        score_letter = chr(65 + cols.index(score_col))
    else:
        # Якщо стовпець Бали не знайдено в заголовках, використовуємо L як стандарт
        score_letter = 'L'

    res_idx = len(cols)
    header_fmt = workbook.add_format({'bold': True, 'bg_color': '#D9EAD3', 'border': 1})
    worksheet.write(0, res_idx, "Загальна сума (Excel)", header_fmt)

    # Вставка формули
    for row_num in range(1, len(df) + 1):
        name_cell = f"{name_letter}{row_num + 1}"
        formula = f'=SUMIF(${name_letter}:${name_letter}, {name_cell}, ${score_letter}:${score_letter})'
        worksheet.write_formula(row_num, res_idx, formula)

    worksheet.set_column(0, res_idx, 25)
    writer.close()

    print("\n--- УСПІХ! ---")
    print(f"Дані зчитано та оброблено. Файл '{output_file}' готовий.")
    files.download(output_file)

if __name__ == "__main__":
    process_grading_xlsx()

Встановлення необхідної бібліотеки: xlsxwriter...
Встановлення необхідної бібліотеки: python-calamine...
--- ЗАВАНТАЖЕННЯ ФАЙЛУ ---


Saving Оцінки Теорія і практика письмового перекладу цифрових текстів (англійська мова) – 13.05.2026, 18_53.xlsx to Оцінки Теорія і практика письмового перекладу цифрових текстів (англійська мова) – 13.05.2026, 18_53.xlsx
Витягую дані (ігноруючи помилки XML)...

--- УСПІХ! ---
Дані зчитано та оброблено. Файл 'Оброблена_відомість_final.xlsx' готовий.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>